# 02d｜从扩充预训练 610 步严格续训到 1220 步

130U-FinLLM｜130U 金融大语言模型

先完成 02c，确认训练达到扩充阶段累计 610 步，并得到 retention before/after
与最终 `verified: true` 备份。本页使用相同的扩充通用训练数据与原 8K 词表，
恢复现有权重、优化器、学习率调度、随机数状态和数据游标；不会重新初始化模型。
不需要重做 01b 或 02c。

默认先验证累计 610→611 步，再续训到累计 1220 步。本段共新增 610 步，
名义约 9,994,240 token 位置；扩充阶段累计约 19,988,480 个位置，实际有效 tokens
由检查点和报告计算。原 seed 的约 4.91M tokens 单独保留，不混入本阶段计数。
本页延长停止预算，保持原 6104 步学习率调度跨度，不表示 100M 预算已完成。

选择「运行时 → 更改运行时类型 → T4 GPU」，逐格执行。不要并行启动第二个训练进程。
连接断开时先等重新连接、确认原进程状态；需要恢复时从第一格重新执行，
不要删除检查点、取消身份校验或换成随机初始化来绕过报错。

In [1]:
import os, sys, json, shutil, subprocess, importlib
from pathlib import Path

REPOSITORY = "https://github.com/130U/130U-FinLLM.git"
REF = "main"  # 首次解析并显示提交；复现时替换成完整的 40 位提交 SHA。
PROJECT = Path("/content/130U-FinLLM")

def command(*args, cwd=None):
    # Explicitly relay subprocess output into the notebook's captured stdout.
    with subprocess.Popen([str(x) for x in args], cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, process.args)

if len(REF) == 40 and all(c in "0123456789abcdefABCDEF" for c in REF):
    REVISION = REF.lower()
else:
    refs = subprocess.check_output(["git", "ls-remote", REPOSITORY, REF], text=True).strip().splitlines()
    if len(refs) != 1:
        raise RuntimeError("REF 不能唯一解析；请填写完整提交 SHA。")
    REVISION = refs[0].split()[0]

if not PROJECT.exists():
    command("git", "clone", "--no-checkout", REPOSITORY, PROJECT)
else:
    remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=PROJECT, text=True).strip()
    if remote.rstrip("/").removesuffix(".git") != REPOSITORY.removesuffix(".git"):
        raise RuntimeError("/content/130U-FinLLM 已属于其他仓库，请选择新目录。")
command("git", "fetch", "--depth", "1", "origin", REVISION, cwd=PROJECT)
command("git", "checkout", "--detach", REVISION, cwd=PROJECT)
os.chdir(PROJECT)
print("本次固定代码提交：", REVISION)

# 保留 Colab 自带的 CUDA / PyTorch；此处仅安装项目的其他依赖。
import torch
from importlib import metadata
from packaging.requirements import Requirement

def tokenizer_install_requirements():
    # Colab can include Transformers even though this project does not use it.
    # Pass its active tokenizer bounds to pip so an old runtime can retain 0.22,
    # while a current runtime (or an already broken one) can use 0.23.
    requirements = ["tokenizers>=0.21,<0.24"]
    try:
        existing = metadata.requires("transformers") or []
    except metadata.PackageNotFoundError:
        existing = []
    for value in existing:
        requirement = Requirement(value)
        if requirement.name.lower().replace("_", "-") == "tokenizers":
            if requirement.marker is None or requirement.marker.evaluate({"extra": ""}):
                requirements.append(str(requirement))
    return list(dict.fromkeys(requirements))

def check_tokenizer_compatibility(requirements):
    installed = metadata.version("tokenizers")
    incompatible = [value for value in requirements
                    if installed not in Requirement(value).specifier]
    if incompatible:
        raise RuntimeError(f"tokenizers {installed} 与现有环境不兼容：{incompatible}")
    print("tokenizers 兼容性检查通过：", installed)

def check_project_dependencies():
    # Read the refreshed installed metadata, not the previous editable checkout.
    problems = []
    for value in metadata.requires("finllm-130u") or []:
        requirement = Requirement(value)
        if requirement.marker is not None and not requirement.marker.evaluate({"extra": ""}):
            continue
        try:
            installed = metadata.version(requirement.name)
        except metadata.PackageNotFoundError:
            problems.append(f"缺少 {requirement.name}")
            continue
        if installed not in requirement.specifier:
            problems.append(f"{requirement.name}=={installed} 不满足 {requirement}")
    if problems:
        raise RuntimeError("项目依赖检查失败：" + "; ".join(problems))
    print("项目已安装依赖检查通过。")

def activate_project_in_kernel():
    # pip installs the editable .pth for NEW Python processes. A notebook kernel
    # that was already running does not process that .pth again automatically.
    source = (PROJECT / "src").resolve()
    expected = source / "finllm"
    if not (expected / "__init__.py").is_file():
        raise RuntimeError(f"缺少项目 Python 源码：{expected}")
    sys.path[:] = [str(source)] + [entry for entry in sys.path if entry != str(source)]
    # Git may have changed code in the same directory since an earlier cell run.
    # Refresh only our modules; never reload Torch or its live CUDA state.
    for name in tuple(sys.modules):
        if name == "finllm" or name.startswith("finllm."):
            del sys.modules[name]
    importlib.invalidate_caches()
    package = importlib.import_module("finllm")
    runtime = importlib.import_module("finllm.runtime")
    for module, filename in ((package, "__init__.py"), (runtime, "runtime.py")):
        if Path(module.__file__).resolve() != expected / filename:
            raise RuntimeError(f"当前 Notebook 导入了错误的项目位置：{module.__file__}")
    if not callable(runtime.restore_seed_checkpoint):
        raise RuntimeError("当前 Notebook 的检查点恢复接口不可用。")
    print("当前 Notebook 内核导入检查通过：", runtime.__file__)

def check_fresh_process_import():
    # Verify the installed editable package separately, without injecting src
    # into PYTHONPATH. Both the current kernel and future CLI processes must work.
    probe = """import json, sys
from pathlib import Path
import finllm
import finllm.runtime as runtime
expected = Path(sys.argv[1]).resolve() / 'finllm'
for module, filename in ((finllm, '__init__.py'), (runtime, 'runtime.py')):
    if Path(module.__file__).resolve() != expected / filename:
        raise RuntimeError('Fresh process imported the wrong project: ' + str(module.__file__))
if not callable(runtime.restore_seed_checkpoint):
    raise RuntimeError('Fresh process checkpoint restore is unavailable')
print(json.dumps({'finllm': str(Path(finllm.__file__).resolve()), 'runtime': str(Path(runtime.__file__).resolve())}))
"""
    result = subprocess.check_output([sys.executable, "-c", probe, str(PROJECT / "src")],
                                     cwd=PROJECT, text=True)
    print("新 Python 进程导入检查通过：", result.strip())

if tuple(int(v) for v in torch.__version__.split("+")[0].split(".")[:2]) < (2, 5):
    raise RuntimeError("需要 PyTorch >= 2.5；请更换较新的 Colab 运行环境。")
tokenizer_requirements = tokenizer_install_requirements()
print("本次 tokenizers 约束：", tokenizer_requirements)
torch_version_before = metadata.version("torch")
torch_file_before = Path(torch.__file__).resolve()
# One resolver transaction replaces the previous finllm metadata AND its
# dependencies. Upgrading tokenizers first leaves old finllm bounds active and
# produces a misleading conflict warning even when the eventual install works.
# The exact installed Torch build is constrained, so pip keeps Colab's CUDA build.
command(sys.executable, "-m", "pip", "install", "-q", "-e", ".",
        "torch==" + torch_version_before, *tokenizer_requirements)
importlib.invalidate_caches()
if metadata.version("torch") != torch_version_before or Path(torch.__file__).resolve() != torch_file_before:
    raise RuntimeError("PyTorch 环境意外变化，请保留输出并停止训练。")
check_tokenizer_compatibility(tokenizer_requirements)
check_project_dependencies()
activate_project_in_kernel()
check_fresh_process_import()
print("Notebook 环境准备完成；PyTorch 保持：", torch_version_before)

def finllm(*args):
    command(sys.executable, "-u", "-m", "finllm", *args, cwd=PROJECT)

def require(*paths):
    missing = [str(p) for p in paths if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("缺少阶段输入，请先准备或从 Drive 恢复：\n" + "\n".join(missing))

finllm("environment")
disk = shutil.disk_usage("/content")
print(f"/content 剩余磁盘：{disk.free / 2**30:.2f} GiB")
Path("runtime_revision.txt").write_text(REVISION + "\n")

Cloning into '/content/130U-FinLLM'...
From https://github.com/130U/130U-FinLLM
 * branch            661a7cab195054cde8ea3a9a841a1565978a7246 -> FETCH_HEAD
HEAD is now at 661a7ca Add strict expanded pretraining continuation notebook
本次固定代码提交： 661a7cab195054cde8ea3a9a841a1565978a7246
本次 tokenizers 约束： ['tokenizers>=0.21,<0.24', 'tokenizers<0.24.0,>=0.23.1']
tokenizers 兼容性检查通过： 0.23.2
项目已安装依赖检查通过。
当前 Notebook 内核导入检查通过： /content/130U-FinLLM/src/finllm/runtime.py
新 Python 进程导入检查通过： {"finllm": "/content/130U-FinLLM/src/finllm/__init__.py", "runtime": "/content/130U-FinLLM/src/finllm/runtime.py"}
Notebook 环境准备完成；PyTorch 保持： 2.11.0+cu130
{
  "python": "3.13.15",
  "platform": "Linux",
  "packages": {
    "torch": "2.11.0+cu130",
    "numpy": "2.1.3",
    "tokenizers": "0.23.2",
    "scikit-learn": "1.6.1",
    "datasets": "4.8.5"
  },
  "git_commit": "661a7cab195054cde8ea3a9a841a1565978a7246",
  "cuda_available": true,
  "cuda": "13.0",
  "gpu": "Tesla T4",
  "gpu_total_bytes": 15637086208,
 

41

In [2]:
MOUNT_DRIVE = True
if not MOUNT_DRIVE:
    raise RuntimeError("本阶段需要已有种子输入及版本化备份，请完成 Drive 挂载。")
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/130U-FinLLM")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive 项目目录：", DRIVE_ROOT)

Mounted at /content/drive
Drive 项目目录： /content/drive/MyDrive/130U-FinLLM


In [3]:
CONTINUATION_RETENTION_BEFORE_SUCCEEDED = False
CONTINUATION_PROBE_SUCCEEDED = False
CONTINUATION_TRAIN_SUCCEEDED = False
CONTINUATION_RETENTION_AFTER_SUCCEEDED = False
CONTINUATION_BACKED_UP = False
SEED_VERSION = "seed_v1"
EXPANSION_VERSION = "expansion_v1"
COMPLETED_UPDATES = 610
PROBE_UPDATES = 611
TARGET_UPDATES = 1220
if not torch.cuda.is_available():
    raise RuntimeError("请选择 T4 GPU 运行时；正式续训不在 CPU 上启动。")
from finllm.expansion_continuation import (
    prepare_expansion_continuation, run_expansion_continuation,
    retention_continuation_check, backup_expansion_continuation)
continuation = prepare_expansion_continuation(
    PROJECT, DRIVE_ROOT, EXPANSION_VERSION, SEED_VERSION,
    completed_updates=COMPLETED_UPDATES, probe_updates=PROBE_UPDATES,
    target_updates=TARGET_UPDATES)
print("已恢复扩充阶段累计步数：", continuation["restored_updates"])
print("已保留 610 步原始成果：", continuation["predecessor_archive"])
print("本次严格续训配置：", json.dumps(continuation["config"], indent=2, ensure_ascii=False))
print("GPU:", torch.cuda.get_device_name(0))

已恢复扩充阶段累计步数： 610
已保留 610 步原始成果： /content/drive/MyDrive/130U-FinLLM/expansions/expansion_v1/continuations/updates_000610_to_001220/baseline_run
本次严格续训配置： {
  "mode": "formal",
  "stage": "pretrain",
  "model": {
    "vocab_size": 8192,
    "hidden_size": 512,
    "num_hidden_layers": 6,
    "num_attention_heads": 8,
    "num_key_value_heads": 4,
    "intermediate_size": 1664,
    "max_position_embeddings": 512,
    "rms_norm_eps": 1e-06,
    "rope_theta": 1000000.0,
    "dropout": 0.0,
    "qk_norm": true,
    "tie_word_embeddings": true
  },
  "tokenizer": "/content/130U-FinLLM/tokenizer/seed_v1",
  "data": "/content/130U-FinLLM/data/processed/expansion_v1/general/manifest.json",
  "validation_data": "/content/130U-FinLLM/data/processed/expansion_v1/general_validation/manifest.json",
  "output_dir": "/content/130U-FinLLM/runs/expansion_v1_pretrain",
  "backup_dir": "/content/drive/MyDrive/130U-FinLLM/live_checkpoints/expansion_v1_pretrain",
  "init_checkpoint": "/content/130U-FinLLM/ch

先读取已冻结 610 步检查点在原 seed 通用验证位置上的训练前诊断，再于累计 1220 步
使用相同 tokenizer、数据及位置测量。另行保留扩充验证集 loss。两者均衡量文本预测，
不能当作看多／看空准确率。已有输入身份完全一致的测量时复用原始报告。

In [4]:
CONTINUATION_RETENTION_BEFORE_SUCCEEDED = False
CONTINUATION_PROBE_SUCCEEDED = False
CONTINUATION_TRAIN_SUCCEEDED = False
CONTINUATION_RETENTION_AFTER_SUCCEEDED = False
CONTINUATION_BACKED_UP = False
continuation_retention_before = retention_continuation_check(continuation, "before")
print(json.dumps(continuation_retention_before, indent=2, ensure_ascii=False))
CONTINUATION_RETENTION_BEFORE_SUCCEEDED = True

{
  "phase": "before",
  "inputs": {
    "checkpoint_sha256": "dbc9ad909108b96443befd49f7c41e5d2f995cf24e1216e45111d222c40ee47c",
    "data_sha256": "f9e89e89893368f157f1d0668e766083ffbe2e2706a3642a61b422dd153c5747",
    "tokenizer_sha256": "3930526cf6d487291cca8ee5f027c0ebeb9a40defb3711110653a36c000ade8a",
    "batch_size": 4,
    "seq_len": 512,
    "max_batches": 50,
    "requested_device": "auto",
    "device": "cuda:0",
    "device_type": "cuda",
    "amp": true,
    "torch_version": "2.11.0+cu130",
    "cuda_version": "13.0"
  },
  "validation": {
    "loss": 4.40319363117218,
    "supervised_tokens": 102400.0
  },
  "completed_updates": 610,
  "scope": "Frozen seed general-validation subset; not a financial stance score"
}


In [5]:
CONTINUATION_PROBE_SUCCEEDED = False
CONTINUATION_TRAIN_SUCCEEDED = False
CONTINUATION_RETENTION_AFTER_SUCCEEDED = False
CONTINUATION_BACKED_UP = False
if not globals().get("CONTINUATION_RETENTION_BEFORE_SUCCEEDED", False):
    raise RuntimeError("请先成功完成 610 步检查点的原验证集诊断。")
continuation_probe = run_expansion_continuation(continuation, "probe", finllm)
print(json.dumps(continuation_probe, indent=2, ensure_ascii=False))
CONTINUATION_PROBE_SUCCEEDED = True

{"stage": "pretrain", "updates": 611, "attempted_updates": 611, "supervised_tokens": 10010624, "loss": 4.2474106550216675, "learning_rate": 0.000294319650902118, "amp_skipped": false}
{
  "stage": "pretrain",
  "mode": "formal",
  "updates": 611,
  "attempted_updates": 611,
  "tokens": 10010624,
  "successful_tokens": 10010624,
  "input_tokens": 10010624,
  "token_definition": "nonpadding supervised positions; one chosen label per DPO pair",
  "latest_checkpoint": "/content/130U-FinLLM/runs/expansion_v1_pretrain/latest.pt",
  "best_checkpoint": "/content/130U-FinLLM/runs/expansion_v1_pretrain/best.pt",
  "loss": 4.2474106550216675,
  "validation": {
    "loss": 4.1299389505386355,
    "supervised_tokens": 102400.0
  },
  "elapsed_seconds": 44.27403583399996,
  "tokens_per_second": 370.05887742942133,
  "peak_vram_bytes": 1141189120,
  "num_parameters": 24255744,
  "epoch": 0,
  "cursor": 19552,
  "completed_epochs": 0,
  "max_epochs": null,
  "effective_max_tokens": 100000000,
  "backu

单步恢复成功后才运行到扩充阶段累计 1220 步。已有目标范围内更靠后的同版本检查点时按实际游标
恢复，不回退到 610 步。每 10 步保存并同步 Drive；原 610 步成果、检查点身份和
retention 报告保存在独立归档。各次调用耗时与吞吐单独记录，重复完成阶段保留原测量。
数据／tokenizer／配置／运行时不兼容会明确停止；停止预算可以增加，调度语义保持不变。

In [6]:
CONTINUATION_TRAIN_SUCCEEDED = False
CONTINUATION_RETENTION_AFTER_SUCCEEDED = False
CONTINUATION_BACKED_UP = False
if not globals().get("CONTINUATION_RETENTION_BEFORE_SUCCEEDED", False) or not globals().get("CONTINUATION_PROBE_SUCCEEDED", False):
    raise RuntimeError("请先完成原验证集诊断和单步严格恢复验证。")
continuation_result = run_expansion_continuation(continuation, "continue", finllm)
print(json.dumps(continuation_result, indent=2, ensure_ascii=False))
CONTINUATION_TRAIN_SUCCEEDED = True

{"stage": "pretrain", "updates": 612, "attempted_updates": 612, "supervised_tokens": 10027008, "loss": 4.060849696397781, "learning_rate": 0.0002942995259160664, "amp_skipped": false}
{"stage": "pretrain", "updates": 613, "attempted_updates": 613, "supervised_tokens": 10043392, "loss": 4.316304683685303, "learning_rate": 0.00029427936611137233, "amp_skipped": false}
{"stage": "pretrain", "updates": 614, "attempted_updates": 614, "supervised_tokens": 10059776, "loss": 4.176645874977112, "learning_rate": 0.0002942591714934645, "amp_skipped": false}
{"stage": "pretrain", "updates": 615, "attempted_updates": 615, "supervised_tokens": 10076160, "loss": 4.318126857280731, "learning_rate": 0.0002942389420677811, "amp_skipped": false}
{"stage": "pretrain", "updates": 616, "attempted_updates": 616, "supervised_tokens": 10092544, "loss": 4.252356290817261, "learning_rate": 0.0002942186778397696, "amp_skipped": false}
{"stage": "pretrain", "updates": 617, "attempted_updates": 617, "supervised_tok

In [7]:
CONTINUATION_RETENTION_AFTER_SUCCEEDED = False
CONTINUATION_BACKED_UP = False
if not globals().get("CONTINUATION_TRAIN_SUCCEEDED", False):
    raise RuntimeError("本段续训尚未成功结束；保留已有备份。")
continuation_retention_after = retention_continuation_check(continuation, "after")
print(json.dumps(continuation_retention_after, indent=2, ensure_ascii=False))
CONTINUATION_RETENTION_AFTER_SUCCEEDED = True

{
  "phase": "after",
  "inputs": {
    "checkpoint_sha256": "3600a2a686faa07ee00b8b40785fa0810821f4bb4cf8bd52d00d9ddc4bc5e461",
    "data_sha256": "f9e89e89893368f157f1d0668e766083ffbe2e2706a3642a61b422dd153c5747",
    "tokenizer_sha256": "3930526cf6d487291cca8ee5f027c0ebeb9a40defb3711110653a36c000ade8a",
    "batch_size": 4,
    "seq_len": 512,
    "max_batches": 50,
    "requested_device": "auto",
    "device": "cuda:0",
    "device_type": "cuda",
    "amp": true,
    "torch_version": "2.11.0+cu130",
    "cuda_version": "13.0"
  },
  "validation": {
    "loss": 3.984176821708679,
    "supervised_tokens": 102400.0
  },
  "completed_updates": 1220,
  "scope": "Frozen seed general-validation subset; not a financial stance score"
}


In [8]:
CONTINUATION_BACKED_UP = False
if not all(globals().get(flag, False) for flag in (
    "CONTINUATION_RETENTION_BEFORE_SUCCEEDED", "CONTINUATION_PROBE_SUCCEEDED",
    "CONTINUATION_TRAIN_SUCCEEDED", "CONTINUATION_RETENTION_AFTER_SUCCEEDED")):
    raise RuntimeError("诊断／单步验证／续训尚未全部成功；不得显示完整阶段备份。")
continuation_backup = backup_expansion_continuation(continuation)
print(json.dumps(continuation_backup, indent=2, ensure_ascii=False))
shutil.copy2(PROJECT / "runtime_revision.txt", continuation["continuation_drive"] / "runtime_revision.txt")
CONTINUATION_BACKED_UP = True
print("扩充通用续训与校验备份完成：", continuation["paths"]["drive"] / "run")
print("本段报告与诊断目录：", continuation["continuation_drive"])
print("610 步原始成果归档：", continuation["predecessor_archive"])
print("原 seed 数据与 300 步成果保留：", continuation["paths"]["seed_drive"])

{
  "files": 7,
  "bytes": 616109865,
  "verified": true,
  "working_copy": false
}
扩充通用续训与校验备份完成： /content/drive/MyDrive/130U-FinLLM/expansions/expansion_v1/run
本段报告与诊断目录： /content/drive/MyDrive/130U-FinLLM/expansions/expansion_v1/continuations/updates_000610_to_001220
610 步原始成果归档： /content/drive/MyDrive/130U-FinLLM/expansions/expansion_v1/continuations/updates_000610_to_001220/baseline_run
原 seed 数据与 300 步成果保留： /content/drive/MyDrive/130U-FinLLM/seeds/seed_v1


完成标准：新单步与正式段报告、累计 1220 步、实际新增／累计 tokens、
retention before/after，以及 `verified: true` 最终备份。保留这些输出用于下一阶段判断。
进入本段后使用 02d 恢复进度；原 02c 的 610 步报告及测量作为独立归档保留。
本页继续通用预训练，未启动金融领域训练、SFT 或 DPO；资产立场与评分能力尚未评估。